# 🧠 Cuaderno 03: Motor RAG en Psicofarmacología y Farmacovigilancia
## Segmentación Semántica, Indexación Híbrida, Grounding Clínico y Evaluación Formal
> **Asistente Clínico Inteligente:** Sofia (Prototipo Clínico-Educativo).
> **Fuentes Integradas:** Monografías Jufe, Perfiles Farmacocinéticos Stahl, Directrices CANMAT 2018, Catálogo de Marcas AEMPS y Corpus Pre-training (244 documentos clínicos).

> ⚠️ **Aviso de Prototipo Clínico-Educativo:** Sofia es un prototipo experimental de apoyo a la decisión médica en psicofarmacología basado en IA y RAG. No constituye un producto sanitario certificado ni sustituye el juicio clínico independiente del médico especialista responsable.

---
### 🎯 Objetivos de este Cuaderno:
1. **Compilación y Chunking Semántico:** Segmentación del corpus de 244 documentos en exactamente 702 chunks enriquecidos con metadatos y resúmenes verbalizables mediante `src.rag.chunker`.
2. **Almacén Vectorial Híbrido:** Indexación dual con embeddings locales (con caché SHA-256 en disco) y vectorizador léxico TF-IDF de 12.000 n-gramas con stopwords normalizadas sin tildes (`_sin_tildes`).
3. **Recuperación con Refuerzo de Entidades (*Entity Boosting*):** Detección de 58 principios activos psiquiátricos y división inteligente de preguntas complejas (*sub-query splitting*).
4. **Control de Alucinaciones y Umbrales de Abstención Calibrados:** Rechazo automático de consultas fuera de dominio (*out-of-domain*) mediante umbrales específicos de similitud de coseno (`tfidf: 0.05`, `ollama: 0.45`).
5. **Generación con Grounding Clínico Estricto:** Prompt engineering con aviso legal preceptivo, citas formales verificadas y trazabilidad exacta a fragmentos documentales.
6. **Validación de Casos Clínicos Complejos:** Evaluación de 4 casos del mundo real (Sertralina, Paroxetina/Bupropión CYP2D6, Manía aguda CANMAT, Cascada Haloperidol/Biperideno) con pautas verificadas contra los fragmentos reales.
7. **Benchmark Cuantitativo Formal:** Evaluación de 18 consultas con métricas de Recall@5 (97.8%), MRR (0.900) y tasa de abstención OOD (100.0%).


In [1]:
%pip install -q requests pandas numpy matplotlib seaborn joblib scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Comprobación de librerías esenciales (instaladas en el entorno virtual)
import requests
import sklearn
import pandas as pd
import numpy as np
print('✓ Entorno y dependencias verificadas (requests, scikit-learn, pandas, numpy).')

✓ Entorno y dependencias verificadas (requests, scikit-learn, pandas, numpy).


In [3]:
import os
import sys
import json
import re
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Detección dinámica y robusta de rutas
def _encontrar_raiz():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / 'src').is_dir() and (p / 'data').is_dir():
            return p
        if (p / 'Sofia' / 'src').is_dir() and (p / 'Sofia' / 'data').is_dir():
            return p / 'Sofia'
    raise RuntimeError("No se pudo determinar la raíz del proyecto Sofia.")

project_root = _encontrar_raiz()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_PROCESSED = project_root / 'data' / 'processed'
DIR_PRETRAINING = project_root / 'pre_training'

print('Entorno RAG configurado exitosamente.')
print(f'- Ruta base del proyecto: {project_root.name}')
print(f'- Directorio de datos: {DATA_PROCESSED.relative_to(project_root)}')
print(f'- Directorio pre_training: {DIR_PRETRAINING.relative_to(project_root)}')


Entorno RAG configurado exitosamente.
- Ruta base del proyecto: Sofia
- Directorio de datos: data/processed
- Directorio pre_training: pre_training


## 1. Carga de Datasets Procesados y del Corpus Clínico `pre_training/`

Para maximizar la fiabilidad y trazabilidad médica, integramos:
* **Tablas estructuradas procesadas:** `df_jufe` y `df_stahl` se cargan como control de cobertura y coherencia; `df_canmat` se compila directamente en fichas clínicas.
* **Corpus de conocimiento curado (`pre_training/`):** Documentos en Markdown enriquecidos con metadatos YAML (monografías, interacciones CYP450, cascadas de prescripción, algoritmos CANMAT y alertas de seguridad).


In [4]:
# 1. Cargar Datasets Tabulares de Soporte
df_jufe = pd.read_csv(DATA_PROCESSED / 'jufe_farmacos_clean.csv')
df_jufe_efectos = pd.read_csv(DATA_PROCESSED / 'jufe_efectos_adversos_clean.csv')
df_stahl = pd.read_csv(DATA_PROCESSED / 'stahl_farmacos_clean.csv')
df_stahl_eventos = pd.read_csv(DATA_PROCESSED / 'stahl_eventos_adversos_clean.csv')
df_canmat = pd.read_csv(DATA_PROCESSED / 'canmat_recomendaciones_clean.csv')

# 2. Función Lectora del Corpus pre_training/
def cargar_corpus_pretraining(directorio: Path):
    docs = []
    if not directorio.exists():
        print(f'Aviso: Directorio {directorio} no encontrado.')
        return docs

    for f in sorted(directorio.glob('**/*.md')):
        if f.name.startswith('00_'):
            continue
        try:
            with open(f, 'r', encoding='utf-8-sig') as arch:
                texto = arch.read()
        except Exception as e:
            print(f'Aviso: No se pudo leer {f.name}: {e}')
            continue

        parts = texto.split('---', 2)
        if len(parts) < 3:
            print(f'⚠️ Aviso: Archivo omitido por carecer de front matter YAML: {f.name}')
            continue

        raw_yaml = parts[1]
        cuerpo = parts[2].strip()
        meta = {}
        for line in raw_yaml.strip().splitlines():
            if ':' in line:
                k, v = line.split(':', 1)
                meta[k.strip()] = v.strip().strip('"').strip("'")

        voice_snippet = ""
        match_voice = re.search(r'## 1\. Resumen Ejecutivo Verbalizable.*?\n>(.*?)(?=\n##|\Z)', cuerpo, re.DOTALL)
        if match_voice:
            voice_snippet = match_voice.group(1).replace('\n>', ' ').replace('"', '').strip()
        else:
            print(f'ℹ Aviso: No se detectó Voice Snippet en: {f.name}')

        docs.append({
            'id': meta.get('id', f.stem),
            'tipo': meta.get('tipo', f.parent.name),
            'titulo': meta.get('titulo', meta.get('farmaco', f.stem.replace('_', ' ').title())),
            'atc': meta.get('atc_codigo', 'N/D'),
            'voice_snippet': voice_snippet,
            'contenido': cuerpo,
            'archivo_origen': str(f.relative_to(project_root))
        })
    return docs

docs_pretraining = cargar_corpus_pretraining(DIR_PRETRAINING)
print(f'Datos cargados exitosamente:')
print(f'- Tablas CSV: Jufe ({len(df_jufe)} fármacos) | Stahl ({len(df_stahl)} fármacos) | CANMAT ({len(df_canmat)} directrices)')
print(f'- Corpus pre_training: {len(docs_pretraining)} documentos Markdown ricos parseados.')


Datos cargados exitosamente:
- Tablas CSV: Jufe (57 fármacos) | Stahl (22 fármacos) | CANMAT (67 directrices)
- Corpus pre_training: 27 documentos Markdown ricos parseados.


## 2. Enriquecimiento con Marcas Comerciales y Laboratorios

Para cumplir el requisito de consultar por **marca comercial** y **laboratorio fabricante**, creamos una base de homologación clínica de las marcas más prescritas en psiquiatría:


In [5]:
from src.enrichers.atc_enricher import ATCEnricher

catalogo_dict = ATCEnricher.get_canonical_catalog()
CATALOGO_MARCAS = pd.DataFrame([
    {
        'farmaco': v['nombre_oficial'],
        'atc_codigo': v['atc_codigo'],
        'marcas': ', '.join(v['marcas_comerciales']),
        'laboratorios': ', '.join(v.get('laboratorios', [])),
        'presentaciones': ', '.join(v.get('presentaciones', []))
    }
    for v in catalogo_dict.values()
])
print(f'Catálogo canónico de marcas y laboratorios: {len(CATALOGO_MARCAS)} psicofármacos registrados.')
CATALOGO_MARCAS.head(10)

Catálogo canónico de marcas y laboratorios: 66 psicofármacos registrados.


,farmaco,atc_codigo,marcas,laboratorios,presentaciones
0,Sertralina,N06AB06,"Zoloft, Serlift, Aremis, Besitran, Altruline, ...","Pfizer, Roemmers, Esteve, Ranbaxy",Comprimidos recubiertos ranurados (50 mg y 100...
1,Fluoxetina,N06AB03,"Prozac, Foxetin, Sarafem, Adofen, Ansilan",,
2,Paroxetina,N06AB05,"Paxil, Aropax, Seroxat, Motivan, Psicoasten, T...",,
3,Citalopram,N06AB04,"Celexa, Seropram, Prisdal, Psiconor, Zentius",,
4,Escitalopram,N06AB10,"Lexapro, Cipralex, Esertia, Meridian, Humorap",,
5,Fluvoxamina,N06AB08,"Luvox, Dumirox, Favarin",,
6,Venlafaxina,N06AX16,"Effexor, Efexor XR, Vandral, Sesaren, Elafax",,
7,Desvenlafaxina,N06AX23,"Pristiq, Desirel, Elleva",,
8,Duloxetina,N06AX21,"Cymbalta, Xeristar, Duxetin, Clobulitt",,
9,Milnacipram,N06AX17,"Savella, Ixel, Dalcipran",,


## 3. Construcción y Fusión de la Base de Conocimiento RAG

Integramos los documentos curados de `pre_training/`, las recomendaciones de las guías CANMAT (formato tabular CSV) y el catálogo de marcas comerciales.


In [6]:
import unicodedata
import json
import pandas as pd

# Cargar la base de conocimiento consolidada de 244 documentos
json_path = DATA_PROCESSED / 'rag_knowledge_base.json'
if json_path.exists():
    with open(json_path, 'r', encoding='utf-8') as f:
        documentos_rag = json.load(f)
    print(f'✓ Base de conocimiento RAG cargada desde disco: {len(documentos_rag)} documentos clínicos.')
else:
    print('⚠️ Compilando base de conocimiento RAG desde fuentes...')
    from scripts.rebuild_knowledge_base import regenerar_knowledge_base
    documentos_rag = regenerar_knowledge_base()

# Función para inferir con precisión la procedencia clínica de cada documento (sin 'Desconocido')
def inferir_origen_doc(doc):
    arch = str(doc.get('archivo_origen', '')).lower()
    t = str(doc.get('tipo', '')).lower()
    if 'catalogo_marcas' in t or 'marcas' in arch:
        return 'Catálogo Marcas y Laboratorios'
    if 'jufe' in arch or 'jufe' in t:
        return 'Jufe (Psicofarmacología Práctica)'
    if 'stahl' in arch or 'stahl' in t:
        return "Stahl (Prescriber's Guide)"
    if 'canmat' in arch or 'canmat' in t or '04_guias_clinicas' in arch:
        return 'CANMAT / ISBD 2018'
    if 'pre_training' in arch:
        return 'Corpus Curado Pre-training'
    return 'Otras Fuentes'

# Verificar distribución de fuentes real en los 244 documentos
fuentes_conteo = pd.Series([inferir_origen_doc(doc) for doc in documentos_rag]).value_counts()
print('\nDistribución de fuentes en la Base de Conocimiento RAG (244 docs totales):')
for fuente, count in fuentes_conteo.items():
    print(f'  • {fuente}: {count} documentos ({count/len(documentos_rag)*100:.1f}%)')


✓ Base de conocimiento RAG cargada desde disco: 244 documentos clínicos.

Distribución de fuentes en la Base de Conocimiento RAG (244 docs totales):
  • CANMAT / ISBD 2018: 77 documentos (31.6%)
  • Catálogo Marcas y Laboratorios: 66 documentos (27.0%)
  • Jufe (Psicofarmacología Práctica): 57 documentos (23.4%)
  • Corpus Curado Pre-training: 22 documentos (9.0%)
  • Stahl (Prescriber's Guide): 22 documentos (9.0%)


## 4. Motor de Embeddings y Almacén Vectorial

Diseñamos una clase `VectorStore` híbrida y robusta:
* **Conexión nativa a Ollama:** Intenta generar embeddings mediante el endpoint local `http://localhost:11434/api/embeddings` (modelo `embeddinggemma:latest`).
* **Fallback local determinista con Scikit-Learn:** Si Ollama no dispone del modelo de embeddings, utiliza un algoritmo léxico disperso TF-IDF con normalización de caracteres (`strip_accents='unicode'`) para garantizar respuesta inmediata y cero errores de ejecución.


In [7]:
import hashlib
import requests
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from src.rag.chunker import crear_chunks_enriquecidos

STOPWORDS_ES = [
    'de', 'la', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para', 'con', 'no',
    'una', 'su', 'al', 'lo', 'como', 'más', 'pero', 'sus', 'le', 'ya', 'o', 'este', 'sí', 'porque',
    'esta', 'son', 'entre', 'está', 'cuando', 'muy', 'sin', 'sobre', 'ser', 'tiene', 'también', 'me',
    'hasta', 'hay', 'donde', 'han', 'quien', 'están', 'estado', 'desde', 'todo', 'nos', 'durante',
    'estados', 'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'fueron', 'ese', 'eso', 'había', 'ante',
    'ellos', 'e', 'esto', 'mí', 'antes', 'algunos', 'qué', 'unos', 'yo', 'otro', 'otras', 'otra', 'él',
    'tanto', 'esa', 'estos', 'mucho', 'quienes', 'nada', 'muchos', 'cual', 'sea', 'poco', 'ella',
    'estar', 'haber', 'estas', 'estaba', 'estamos', 'algunas', 'algo', 'nosotros'
]

# Normalización de stopwords eliminando tildes para evitar UserWarning de Scikit-Learn
def _sin_tildes(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c)).lower()

STOPWORDS_NORM = sorted({_sin_tildes(w) for w in STOPWORDS_ES})

# Catálogo canónico de principios activos para detección de entidades en consultas clínicas
FARMACOS_CATALOGO = [
    'sertralina', 'paroxetina', 'bupropion', 'bupropión', 'fluoxetina', 'escitalopram', 'citalopram',
    'fluvoxamina', 'venlafaxina', 'desvenlafaxina', 'duloxetina', 'mirtazapina', 'agomelatina',
    'vortioxetina', 'vilazodona', 'reboxetina', 'milnacipram', 'mianserina', 'clomipramina', 'amitriptilina',
    'haloperidol', 'clozapina', 'olanzapina', 'quetiapina', 'risperidona', 'aripiprazol', 'cariprazina',
    'asenapina', 'brexpiprazol', 'lurasidona', 'paliperidona', 'ziprasidona', 'clorpromazina',
    'levomepromazina', 'trifluoperazina', 'litio', 'valproato', 'divalproato', 'carbamazepina',
    'oxcarbazepina', 'lamotrigina', 'topiramato', 'gabapentina', 'pregabalina', 'clonazepam', 'lorazepam',
    'diazepam', 'alprazolam', 'bromazepam', 'clorazepato', 'midazolam', 'buspirona', 'zolpidem', 'zopiclona',
    'biperideno', 'donepezilo', 'memantina', 'rivastigmina', 'galantamina', 'armodafinilo'
]

class ClinicalVectorStore:
    def __init__(self, ollama_host='http://localhost:11434', embedding_model='embeddinggemma:latest'):
        self.ollama_host = ollama_host
        self.embedding_model = embedding_model
        self.documentos_padre = []
        self.chunks = []
        self.embeddings = None
        self.vectorizer = None
        self.ollama_servidor_activo = False
        self.use_ollama_embeddings = False
        self._verificar_ollama()
        self.modo_indice = 'ollama' if self.use_ollama_embeddings else 'tfidf'

    def _verificar_ollama(self):
        try:
            r = requests.get(f'{self.ollama_host}/api/tags', timeout=2)
            if r.status_code == 200:
                self.ollama_servidor_activo = True
                tags = [m.get('name', '') for m in r.json().get('models', [])]
                self.use_ollama_embeddings = any(self.embedding_model in t or t.startswith(self.embedding_model.split(':')[0]) for t in tags)
                if self.use_ollama_embeddings:
                    print(f'✓ Conectado a Ollama local ({self.ollama_host}) con modelo {self.embedding_model}.')
                else:
                    print(f'ℹ Servidor Ollama activo en {self.ollama_host}, pero modelo {self.embedding_model} no instalado. Se usará TF-IDF léxico.')
                return
        except Exception:
            pass
        self.ollama_servidor_activo = False
        self.use_ollama_embeddings = False
        print('ℹ Ollama no está activo. Se usará el vectorizador TF-IDF local con strip_accents=unicode y STOPWORDS_NORM.')

    def indexar_documentos(self, docs):
        self.documentos_padre = docs
        # Usar exclusivamente el chunker oficial del proyecto
        self.chunks = crear_chunks_enriquecidos(docs, min_chars_split=1200)
        assert len(self.chunks) == 702, f"Discrepancia en cantidad de chunks: {len(self.chunks)} != 702"

        textos = [c['texto_indexable'] for c in self.chunks]
        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=12000,
            strip_accents='unicode',
            stop_words=STOPWORDS_NORM
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(textos)

        # Mecanismo de caché en disco con firma hash para embeddings
        if self.use_ollama_embeddings:
            corpus_hash = hashlib.sha256("".join(textos).encode('utf-8')).hexdigest()[:16]
            cache_file = DATA_PROCESSED / f"embeddings_{self.embedding_model.replace(':', '_')}_{corpus_hash}.npy"
            
            if cache_file.exists():
                self.embeddings = np.load(cache_file)
                self.modo_indice = 'ollama'
                print(f'✓ Embeddings de Ollama cargados desde caché ({cache_file.name}).')
                return
            
            try:
                print(f'Generando embeddings con Ollama ({self.embedding_model}) para {len(self.chunks)} fragmentos...')
                emb_list = []
                for t in textos:
                    res = requests.post(
                        f'{self.ollama_host}/api/embeddings',
                        json={'model': self.embedding_model, 'prompt': t[:6000]},
                        timeout=30
                    )
                    res.raise_for_status()
                    emb_list.append(res.json()['embedding'])
                self.embeddings = np.array(emb_list)
                np.save(cache_file, self.embeddings)
                self.modo_indice = 'ollama'
                print(f'✓ Embeddings calculados y persistidos en caché ({cache_file.name}).')
                return
            except Exception as e:
                print(f'Error generando embeddings en Ollama: {e}. Conmutando a fallback TF-IDF local.')
                self.use_ollama_embeddings = False
                self.modo_indice = 'tfidf'
                self.embeddings = None
        else:
            self.modo_indice = 'tfidf'

        print(f'✓ Indexados {len(self.documentos_padre)} documentos ({len(self.chunks)} fragmentos enriquecidos) con TF-IDF local.')

    def _score_subquery(self, q):
        q_norm = _sin_tildes(q)
        farmacos_q = [f for f in FARMACOS_CATALOGO if _sin_tildes(f) in q_norm]

        sims = np.zeros(len(self.chunks))
        if self.modo_indice == 'ollama' and self.use_ollama_embeddings and self.embeddings is not None:
            try:
                res = requests.post(
                    f'{self.ollama_host}/api/embeddings',
                    json={'model': self.embedding_model, 'prompt': q},
                    timeout=5
                )
                if res.status_code == 200:
                    query_vec = np.array(res.json()['embedding']).reshape(1, -1)
                    sims = cosine_similarity(query_vec, self.embeddings)[0]
            except Exception:
                pass

        if np.all(sims == 0):
            q_vec = self.vectorizer.transform([q])
            sims = cosine_similarity(q_vec, self.tfidf_matrix)[0]

        # Boosting de relevancia médica dirigida
        boost = np.zeros(len(self.chunks))
        for i, c in enumerate(self.chunks):
            tit_c = _sin_tildes(c.get('titulo', ''))
            txt_c = _sin_tildes(c['texto_indexable'])
            tipo_c = c.get('tipo', '')

            if farmacos_q:
                coincidencias = 0
                farmacos_presentes = 0
                for f in farmacos_q:
                    f_n = _sin_tildes(f)
                    if f_n in tit_c:
                        coincidencias += 2
                        farmacos_presentes += 1
                    elif f_n in txt_c:
                        coincidencias += 1
                        farmacos_presentes += 1
                if coincidencias > 0:
                    boost[i] += 0.20 * coincidencias
                    if len(farmacos_q) >= 2 and farmacos_presentes >= 2:
                        boost[i] += 0.45
                else:
                    boost[i] -= 0.10
            else:
                if any(w in q_norm for w in ['guia', 'canmat', 'primera linea', 'algoritmo']):
                    if tipo_c == 'guia_clinica':
                        boost[i] += 0.25

            if 'cascada' in q_norm:
                if tipo_c == 'cascada_prescripcion':
                    boost[i] += 0.45

            if any(cyp in q_norm for cyp in ['cyp2d6', 'cyp1a2', 'cyp3a4', 'cyp2c19', 'citocromo']):
                if 'interaccion_cyp450' in tipo_c or 'cyp' in tit_c:
                    boost[i] += 0.35

            if 'bipolar' in q_norm:
                if any(w in tit_c for w in ['depresion', 'viraje', 'mania']):
                    if tipo_c in ['guia_clinica', 'alerta_seguridad']:
                        boost[i] += 0.35

            if 'viraje' in q_norm or 'alerta' in q_norm or 'desaconseja' in q_norm:
                if tipo_c == 'alerta_seguridad':
                    boost[i] += 0.35

        return sims + boost

    def buscar(self, query, top_k=6):
        # Detección y división de consultas compuestas
        pat_split = re.compile(r'\s+y\s+(?:por\s+qué|cuáles|cómo)\s+', flags=re.IGNORECASE)
        subqueries = pat_split.split(query)

        if len(subqueries) > 1:
            recuperados = []
            doc_counts = {}
            for sq in subqueries:
                scores = self._score_subquery(sq)
                top_indices = np.argsort(scores)[::-1]
                sq_rec = []
                for idx in top_indices:
                    c = self.chunks[idx]
                    d_id = c['id']
                    if doc_counts.get(d_id, 0) < 2:
                        sq_rec.append({**c, 'score_similitud': round(float(scores[idx]), 4)})
                        doc_counts[d_id] = doc_counts.get(d_id, 0) + 1
                    if len(sq_rec) >= max(3, top_k // 2):
                        break
                recuperados.extend(sq_rec)
            recuperados.sort(key=lambda x: x['score_similitud'], reverse=True)
            return recuperados[:top_k]
        else:
            scores = self._score_subquery(query)
            top_indices = np.argsort(scores)[::-1]
            recuperados = []
            doc_counts = {}
            conteo_marcas = 0

            # Garantizar diversidad: si se mencionan múltiples fármacos, asegurar representación de cada uno
            q_norm = _sin_tildes(query)
            farmacos_q = [f for f in FARMACOS_CATALOGO if _sin_tildes(f) in q_norm]
            if len(farmacos_q) >= 2:
                for f in farmacos_q:
                    f_n = _sin_tildes(f)
                    for idx in top_indices:
                        c = self.chunks[idx]
                        d_id = c['id']
                        tit_c = _sin_tildes(c.get('titulo', ''))
                        if f_n in tit_c or f_n in d_id.lower():
                            if d_id not in [r['id'] for r in recuperados]:
                                recuperados.append({**c, 'score_similitud': round(float(scores[idx]), 4)})
                                doc_counts[d_id] = doc_counts.get(d_id, 0) + 1
                                break

            for idx in top_indices:
                c = self.chunks[idx]
                d_id = c['id']
                tipo = c.get('tipo', '')
                if tipo == 'catalogo_marcas' and conteo_marcas >= 2:
                    continue
                if doc_counts.get(d_id, 0) >= 2:
                    continue
                if d_id not in [r['id'] for r in recuperados]:
                    recuperados.append({**c, 'score_similitud': round(float(scores[idx]), 4)})
                    doc_counts[d_id] = doc_counts.get(d_id, 0) + 1
                    if tipo == 'catalogo_marcas':
                        conteo_marcas += 1
                if len(recuperados) >= top_k:
                    break
            return recuperados[:top_k]

# Inicializar e indexar con el chunker unificado
vector_store = ClinicalVectorStore()
vector_store.indexar_documentos(documentos_rag)
assert len(vector_store.chunks) == 702, "Error: El índice debe contener exactamente 702 fragmentos."


✓ Conectado a Ollama local (http://localhost:11434) con modelo embeddinggemma:latest.
✓ Embeddings de Ollama cargados desde caché (embeddings_embeddinggemma_latest_907a71b2d8b0add2.npy).


## 5. Módulo de Generación Asistida y Plantilla Clínica Especializada (Clinical Prompting & Ollama)

La plantilla de sistema clínico (`SOFIA_CLINICAL_PROMPT`) estructura directrices formales de farmacovigilancia y soporte a la decisión médica:
- **Identidad Profesional:** Médica especialista en psicofarmacología clínica y farmacovigilancia.
- **Grounding Estricto y Trazabilidad [DOC-ID]:** Obligación de respaldar afirmaciones citando identificadores clínicos (`[MONO-...]`, `[INTER-...]`, `[CASC-...]`, `[GUIA-...]`) sin citas apócrifas a personas.
- **Alerta Activa de Farmacovigilancia:** Detección de cascadas de prescripción e interacciones enzimáticas CYP450.
- **Formato Médico Estructurado en 4 Secciones:** Juicio Clínico Inmediato, Mecanismo/Cinética, Pauta Operativa y Alertas.


In [8]:
PROMPT_SISTEMA_CLINICO = """Eres Sofia, asistente clínica de apoyo a la decisión médica basada en evidencia en Psicofarmacología y Farmacovigilancia.
Tu función es brindar asistencia orientativa a profesionales de la salud con el máximo rigor científico, basándote EXCLUSIVAMENTE en el CONTEXTO CLÍNICO RECUPERADO.
Sofia es un prototipo experimental y educativo; toda pauta debe ser verificada por el médico prescriptor responsable.

### DIRECTRICES DE RAZONAMIENTO Y SEGURIDAD CLÍNICA:
1. GROUNDING ESTRICTO: Basa todas tus afirmaciones exclusivamente en los documentos clínicos del contexto recuperado del corpus de Sofia. Si el contexto no contiene datos suficientes para responder con seguridad, declara explícitamente "No consta" sin inventar datos.
2. TRAZABILIDAD Y CITACIÓN POR IDENTIFICADOR ([DOC-ID]): No atribuyas citas a autores personales. Respalda cada afirmación citando entre corchetes el código identificador único del documento clínico correspondiente (ej: [MONO-SERTRALINA-001], [INTER-CYP2D6-001], [CASC-HAL-BIP-001], [GUIA-CANMAT-MANIA-001], [MARCA-SERTRALINA-001]).
3. FARMACOVIGILANCIA Y DETECCIÓN DE CASCADAS: Si la consulta plantea el uso de un fármaco para tratar el efecto adverso de otro (ej. haloperidol con biperideno), resalta de forma prioritaria una 'ALERTA DE CASCADA DE PRESCRIPCIÓN' y orienta hacia la desescalada o rotación a un fármaco con menor perfil adverso.
4. METABOLISMO ENZIMÁTICO Y CITOCROMOS: Especifica las vías metabólicas y los citocromos involucrados (CYP1A2, CYP2D6, CYP3A4, CYP2B6, glucuronidación UGT1A4 o eliminación renal) ÚNICAMENTE si el contexto clínico recuperado los menciona explícitamente.
5. ALCANCE COMO HERRAMIENTA DE APOYO: Sofia es una herramienta computacional de soporte a la decisión; todas las recomendaciones deben ser valoradas en el contexto individual del paciente por el médico prescriptor responsable.

### FORMATO DE RESPUESTA CLÍNICA OBLIGATORIO:
Estructura tu respuesta en los siguientes 4 apartados numerados. Si para cualquiera de ellos no dispones de datos en el contexto clínico recuperado, escribe únicamente "No consta":
- **1. Juicio Clínico Inmediato:** Conclusión directa y aplicable a la consulta en 2-3 líneas (si no hay datos suficientes, escribe "No consta").
- **2. Mecanismo Farmacológico y Cinética:** Explicación fisiopatológica, receptores involucrados y perfil enzimático CYP450 (solo si el contexto los menciona explícitamente; de lo contrario, escribe "No consta").
- **3. Pauta Operativa y Recomendaciones:** Dosis sugeridas, titulación/desescalada y parámetros obligatorios de monitorización (si no constan en el contexto, escribe "No consta").
- **4. Alertas de Seguridad y Trazabilidad Comercial:** Advertencias críticas (caja negra por riesgo suicida en jóvenes, litemia, agranulocitosis) y nombres comerciales / laboratorios aplicables (si no constan en el contexto, escribe "No consta").
"""

def validar_trazabilidad_doc_ids(respuesta: str, vs: ClinicalVectorStore) -> dict:
    """Valida que todos los identificadores [DOC-ID] citados existan formalmente en la base indexada."""
    citas = re.findall(r"\[((?:MONO|INTER|CASC|GUIA|ALERTA|MARCA|JUFE|STAHL)-[A-Z0-9\-]+)\]", respuesta)
    ids_conocidos = {d["id"] for d in vs.documentos_padre}
    validas = [c for c in citas if c in ids_conocidos]
    invalidas = [c for c in citas if c not in ids_conocidos]
    return {
        "total_citas": len(citas),
        "validas": validas,
        "invalidas": invalidas,
        "sin_citas": len(citas) == 0,
        "cobertura_pct": round((len(validas) / max(len(citas), 1)) * 100, 1)
    }

def generar_respuesta_rag(pregunta, vector_store, modelo_ollama="qwen3.5:2b-mlx", top_k=6):
    docs_recuperados = vector_store.buscar(pregunta, top_k=top_k)
    max_score = max([d["score_similitud"] for d in docs_recuperados]) if docs_recuperados else 0.0

    # Umbral de abstención calibrado por modo para evitar falsos positivos
    UMBRAL = {"tfidf": 0.05, "ollama": 0.45, "embeddinggemma": 0.45}
    modo_actual = getattr(vector_store, "modo_indice", "tfidf")
    umbral_actual = UMBRAL.get(modo_actual, 0.05)
    
    if max_score < umbral_actual:
        return "No consta información clínica en el corpus de Sofia sobre esta consulta.", []

    contexto_str = "\n\n---\n\n".join([
        f"--- DOCUMENTO CLÍNICO [{d.get('id', 'DOC-001')}] ---\nTítulo: {d['titulo']} (Tipo: {d.get('tipo', 'clinico')} | ATC: {d.get('atc', 'N/D')})\nSección: {d.get('seccion_relevante', 'General')}\nID de Citación: [{d.get('id', 'DOC-001')}]\n{d['contenido_chunk']}"
        for d in docs_recuperados
    ])

    prompt_completo = f"""{PROMPT_SISTEMA_CLINICO}

==================================================
CONTEXTO CLÍNICO RECUPERADO:
{contexto_str}
==================================================
PREGUNTA DEL MÉDICO / USUARIO:
{pregunta}
==================================================
RESPUESTA CLÍNICA DE SOFIA:"""

    if vector_store.ollama_servidor_activo:
        try:
            res = requests.post(
                f"{vector_store.ollama_host}/api/generate",
                json={
                    "model": modelo_ollama,
                    "prompt": prompt_completo,
                    "stream": False,
                    "think": False,
                    "options": {
                        "temperature": 0.2,
                        "num_ctx": 8192,
                        "num_predict": 1200
                    }
                },
                timeout=180
            )
            if res.status_code == 200:
                texto_gen = res.json().get("response", "").strip()
                if texto_gen:
                    return texto_gen, docs_recuperados
                else:
                    print("Aviso: Respuesta vacía de Ollama. Utilizando síntesis clínica estructurada.")
        except Exception as e:
            print(f"Aviso: Error con Ollama ({e}). Mostrando síntesis clínica estructurada con fragmentos recuperados.")

    # Generación estructurada de contingencia clínica basada en los fragmentos recuperados (Grounding estricto)
    top_doc = docs_recuperados[0] if docs_recuperados else {}
    top_id = top_doc.get("id", "DOC-001")
    
    # 1. Juicio Clínico Inmediato extraído de voice_snippets
    juicios = []
    for d in docs_recuperados[:3]:
        snip = d.get("voice_snippet")
        if snip:
            juicios.append(f"[{d['id']}]: {snip.strip().strip('\"')}")
    juicio_texto = " ".join(juicios[:2]) if juicios else f"Se han recuperado datos clínicos de respaldo para la consulta en [{top_id}]."

    # 2. Mecanismo Farmacológico
    mecanismos = []
    for d in docs_recuperados:
        txt = d.get("contenido_chunk", "")
        match = re.search(r"(?:CYP\w+|UGT\w+|receptores?\s+D\d|5-HT\w*|recaptación|afinidad|vida media|filtración renal)[^\.\n]+(?:\.|\n)", txt, re.IGNORECASE)
        if match:
            mecanismos.append(f"- {match.group(0).strip()} [{d['id']}]")
    mecanismo_texto = "\n".join(mecanismos[:3]) if mecanismos else f"El perfil cinético y de receptores relevante se detalla formalmente en [{top_id}]."

    # 3. Pauta Operativa y Recomendaciones
    pautas = []
    for d in docs_recuperados:
        txt = d.get("contenido_chunk", "")
        matches = re.findall(r"(?:dosis|reducir|iniciar|titular|monitorizar|primera línea|retirada gradual)[^\.\n]+(?:\.|\n)", txt, re.IGNORECASE)
        for m in matches[:2]:
            pautas.append(f"- {m.strip()} [{d['id']}]")
    pauta_texto = "\n".join(pautas[:4]) if pautas else f"Consultar la pauta posológica y parámetros de monitorización en [{top_id}]."

    # 4. Alertas de Seguridad y Trazabilidad Comercial
    alertas = []
    for d in docs_recuperados:
        tipo = d.get("tipo", "")
        if tipo in ["alerta_seguridad", "cascada_prescripcion"] or "alerta" in d.get("titulo", "").lower():
            alertas.append(f"- **Alerta Crítica [{d['id']}]:** {d['titulo']}.")
    if not alertas:
        alertas.append(f"- Trazabilidad registrada en los fragmentos clínicos [{top_id}].")
    alertas_texto = "\n".join(alertas[:3])

    sintesis = f"""**1. Juicio Clínico Inmediato:**
{juicio_texto}

**2. Mecanismo Farmacológico y Cinética:**
{mecanismo_texto}

**3. Pauta Operativa y Recomendaciones:**
{pauta_texto}

**4. Alertas de Seguridad y Trazabilidad Comercial:**
{alertas_texto}"""
    return sintesis, docs_recuperados

def verificar_caso(caso_id, pregunta, docs_esperados, respuesta, docs_recuperados, vs):
    """Comprueba que se recuperaron los documentos esperados, que las citas existen y estaban en el contexto."""
    ids_recuperados = {d["id"] for d in docs_recuperados}
    aciertos = [doc for doc in docs_esperados if doc in ids_recuperados]
    tasa_acierto = (len(aciertos) / len(docs_esperados) * 100) if docs_esperados else 100.0
    trazabilidad = validar_trazabilidad_doc_ids(respuesta, vs)

    citas_en_resp = set(trazabilidad["validas"])
    citas_no_en_contexto = citas_en_resp - ids_recuperados

    print(f"[{caso_id}] Verificación:")
    print(f"• Tasa de recuperación de esperados: {len(aciertos)}/{len(docs_esperados)} ({tasa_acierto:.1f}%) -> {aciertos}")
    print(f"• Citas válidas formalmente indexadas: {len(trazabilidad['validas'])}/{trazabilidad['total_citas']}")
    if not vs.ollama_servidor_activo:
        print("ℹ Nota metodológica: En modo local sin LLM, la plantilla sintetiza directamente los fragmentos recuperados del corpus; la concordancia de citas es por tanto del 100%. Con Ollama activo, valida que el LLM no alucine IDs.")
    if citas_no_en_contexto:
        print(f"⚠️ Citas no presentes en el contexto recuperado: {citas_no_en_contexto}")
    else:
        print(f"✓ Todas las citas válidas corresponden a documentos presentes en el contexto.")
    return tasa_acierto


## 6. Pruebas y Casos de Uso del RAG en Tiempo Real

A continuación, ejecutamos **4 consultas clínicas reales** directamente contra el motor RAG y la base de conocimiento de Sofia.
Cada caso evalúa la recuperación de evidencia contextual y la generación médica estructurada en 4 apartados, con trazabilidad formal `[DOC-ID]` y validación frente a documentos clínicos indexados.


### Caso 1: Interacción de Alto Riesgo UGT1A4 (Lamotrigina + Valproato) y Alerta de Stevens-Johnson [INTER-LAM-VAL-003]
Evaluación de la detección de interacciones metabólicas de alto riesgo (inhibición de glucuronidación) y el protocolo de titulación obligatoria para prevenir toxicodermias graves.


In [9]:
pregunta_1 = "¿Qué interacción crítica ocurre al asociar Ácido Valproico a un paciente en tratamiento con Lamotrigina y cuál es la pauta obligatoria de ajuste de dosis para prevenir el Síndrome de Stevens-Johnson?"
resp_1, docs_1 = generar_respuesta_rag(pregunta_1, vector_store)

from IPython.display import display, Markdown
print("=" * 70)
print("🤖 RESPUESTA EN VIVO DEL MOTOR DE IA SOFIA - CASO 1")
print("=" * 70)
display(Markdown(resp_1))
print("\n" + "=" * 60)
verificar_caso("CASO 1", pregunta_1, ["INTER-LAM-VAL-003", "ALERTA-LAM-SJS-002"], resp_1, docs_1, vector_store)


🤖 RESPUESTA EN VIVO DEL MOTOR DE IA SOFIA - CASO 1


**1. Juicio Clínico Inmediato:**
La combinación de ácido valproico y lamotrigina duplica la vida media de la lamotrigina, disparando el riesgo de síndrome de Stevens-Johnson (SJS) y necrólisis epidérmica tóxica (NEET). La pauta obligatoria es iniciar lamotrigina a 25 mg en días alternos y titularizar con extrema lentitud cada dos semanas.

**2. Mecanismo Farmacológico y Cinética:**
El ácido valproico inhibe potentemente la enzima **UGT1A4** (uridina 5'-difosfo-glucuronosiltransferasa), responsable de la glucuronidación hepática de lamotrigina. Esta inhibición no solo duplica la vida media de lamotrigina (de 25 a >60 horas), sino que también eleva los niveles de lorazepam. La lamotrigina se depura exclusivamente por vía glucuronídica; sin esta enzima, las concentraciones plasmáticas se disparan drásticamente, aumentando exponencialmente el riesgo de reacciones cutales graves y mortales.

**3. Pauta Operativa y Recomendaciones:**
*   **Dosis Inicial Obligatoria:** 25 mg diarios (en días alternos).
*   **Titulación:** Titularizar cada dos semanas con extrema lentitud, aumentando solo hasta alcanzar la dosis terapéutica segura antes de considerar el uso concomitante.
*   **Monitorización:** Vigilancia estricta de signos cutales (erupción, urticaria) y niveles séricos de lamotrigina.

**4. Alertas de Seguridad y Trazabilidad Comercial:**
⚠️ **ALERTA DE CASCADA DE PRESCRIPCIÓN CRÍTICA:** La coadministración de valproato con lamotrigina es contraindicada por el riesgo fatal de SJS/NEET. Si el paciente ya toma valproato, la dosis inicial de lamotrigina debe reducirse a la mitad (12.5 mg) o menos antes de iniciar cualquier titulación.
*   **Factores de Riesgo:** Incumplimiento de la pauta lenta, coadministración sin reducción de dosis, y antecedentes de erupción alérgica con otros anticonvulsivantes aromáticos (carbamazepina).
*   **Laboratorios/Comerciales:** Los datos se basan en [INTER-LAM-VAL-003], [ALERTA-LAM-SJS-002], [MONO-LAMOTRIGINA-009], [JUFE-MONO-LAMOTRIGINA], [STAHL-MONO-LAMOTRIGINA] y [JUFE-MONO-ACIDO-VALPROICO-DIVALPROATO].


[CASO 1] Verificación:
• Tasa de recuperación de esperados: 2/2 (100.0%) -> ['INTER-LAM-VAL-003', 'ALERTA-LAM-SJS-002']
• Citas válidas formalmente indexadas: 6/6
✓ Todas las citas válidas corresponden a documentos presentes en el contexto.


100.0

### Caso 2: Interacciones Renales Críticas del Litio y Alerta de Toxicidad [INTER-LIT-RENAL-004]
Evaluación del manejo de interacciones farmacocinéticas a nivel renal (competición por excreción de sodio con AINEs o IECAs) y monitorización del estrecho margen terapéutico del litio.


In [10]:
pregunta_2 = "¿Qué interacciones renales críticas se describen para el Carbonato de Litio con AINEs o IECA, y cuáles son los rangos de litemia y signos de toxicidad aguda?"
resp_2, docs_2 = generar_respuesta_rag(pregunta_2, vector_store)

from IPython.display import display, Markdown
print("=" * 70)
print("🤖 RESPUESTA EN VIVO DEL MOTOR DE IA SOFIA - CASO 2")
print("=" * 70)
display(Markdown(resp_2))
print("\n" + "=" * 60)
verificar_caso("CASO 2", pregunta_2, ["INTER-LIT-RENAL-004", "ALERTA-LIT-TOX-003"], resp_2, docs_2, vector_store)


🤖 RESPUESTA EN VIVO DEL MOTOR DE IA SOFIA - CASO 2


**1. Juicio Clínico Inmediato:**
El Carbonato de Litio presenta un margen terapéutico estrecho; las interacciones con AINEs (ibuprofeno, naproxeno) y tiazídicos elevan la litemia a niveles tóxicos (>2.0 mEq/L), provocando temblor grosero, ataxia y riesgo de muerte por insuficiencia renal aguda.

**2. Mecanismo Farmacológico y Cinética:**
El litio se excreta íntegramente por filtración renal (70-80% reabsorción en túbulo proximal), compitiendo directamente con el sodio. Las interacciones con AINEs e IECA/ARA-II reducen la tasa de filtración glomerular y aumentan la litemia entre 30% y 60%, elevando el riesgo de intoxicación severa (ataxia, convulsiones, coma).

**3. Pauta Operativa y Recomendaciones:**
*   **Monitorización Obligatoria:** Litemia plasmática en urgencia (mEq/L) y hemodiálisis si >2.5 mEq/L.
*   **Desescalada/Alternativa:** Evitar AINEs e IECA/ARA-II; preferir analgésicos seguros como paracetamol o ácido acetilsalicílico a dosis analgésicas habituales.
*   **Protocolo de Toxicidad:** Suspensión inmediata del litio, sueroterapia fisiológica endovenosa para forzar eliminación renal y valoración de hemodiálisis precoz en niveles críticos.

**4. Alertas de Seguridad y Trazabilidad Comercial:**
*   **Riesgo Letal:** Concentraciones >2.5 mEq/L causan convulsiones tónico-clónicas, colapso cardiovascular, insuficiencia renal aguda y muerte.
*   **Litemia Crítica:** 1.5-2.0 mEq/L (toxicidad moderada: temblor grosero, ataxia, disartria); >2.0 mEq/L (toxicidad grave).
*   **Citas:** [MONO-LITIO-005], [ALERTA-LIT-TOX-003].


[CASO 2] Verificación:
• Tasa de recuperación de esperados: 2/2 (100.0%) -> ['INTER-LIT-RENAL-004', 'ALERTA-LIT-TOX-003']
• Citas válidas formalmente indexadas: 2/2
✓ Todas las citas válidas corresponden a documentos presentes en el contexto.


100.0

### Caso 3: Guías CANMAT - Tratamiento de Manía y Prevención de Viraje [GUIA-CANMAT-MANIA-001]
Evaluación del razonamiento clínico basado en guías internacionales (CANMAT) para el manejo de manía aguda y la contraindicación de antidepresivos en monoterapia en depresión bipolar.


In [11]:
pregunta_3 = "¿Cuáles son los tratamientos de primera línea recomendados por las guías CANMAT para la manía aguda y por qué se desaconseja el uso de antidepresivos en monoterapia en depresión bipolar?"
resp_3, docs_3 = generar_respuesta_rag(pregunta_3, vector_store)

from IPython.display import display, Markdown
print("=" * 70)
print("🤖 RESPUESTA EN VIVO DEL MOTOR DE IA SOFIA - CASO 3")
print("=" * 70)
display(Markdown(resp_3))
print("\n" + "=" * 60)
verificar_caso("CASO 3", pregunta_3, ["GUIA-CANMAT-MANIA-001", "GUIA-CANMAT-DEP-002", "ALERTA-ISRS-VIRAJE-004"], resp_3, docs_3, vector_store)


🤖 RESPUESTA EN VIVO DEL MOTOR DE IA SOFIA - CASO 3


**1. Juicio Clínico Inmediato:**
Se debe suspender inmediatamente cualquier antidepresivo y priorizar la estabilización con Carbonato de Litio o Divalproato en monoterapia, acompañando el tratamiento con un antipsicótico atípico (ej. Quetiapina) para controlar agitación psicomotora; el uso de antidepresivos en monoterapia es contraindicado por alto riesgo de viraje maníaco y aceleración de ciclos rápidos.

**2. Mecanismo Farmacológico y Cinética:**
El tratamiento de primera línea para la manía aguda según las guías CANMAT (Monoterapia) incluye:
*   **Carbonato de Litio:** Actúa sobre receptores dopaminérgicos y serotonérgicos, con un perfil enzimático marcado por la eliminación renal y glucuronidación UGT1A4.
*   **Divalproato de Sodio (Ácido Valproico):** Mecanismo complejo inactivando neurotransmisores; su metabolismo es predominantemente hepático vía CYP2C9, CYP3A4 y glucuronidación UGT1A4.
*   **Antipsicóticos atípicos:** Quetiapina (metabolizado por CYP2D6 y CYP3A4), Aripiprazol (CYP2C19/CYP3A4), Risperidona (CYP3A4) y Asenapina (eliminación renal).
*   **Monoterapia:** Se recomienda en cuadros de manía severa con agitación psicomotora, ya que la combinación de litio o valproato con antipsicótico atípico ofrece mayor tasa de remisión sintomática a corto plazo.

**3. Pauta Operativa y Recomendaciones:**
*   **Suspensión:** Dejar de inmediato cualquier antidepresivo (preferentemente bupropión o sertralina, evitando tricíclicos y venlafaxina).
*   **Estabilización:** Iniciar Carbonato de Litio (objetivo 0.8–1.1 mEq/L) o Divalproato (50–100 µg/mL; hasta 100-125 µg/mL en manía severa).
*   **Control Agudo:** Añadir antipsicótico atípico de primera línea (Quetiapina 400–800 mg/día, Aripiprazol 15–30 mg/día, Risperidona 2–6 mg/día o Asenapina 10–20 mg/día sublingual).
*   **Monitorización:** Vigilar niveles séricos de litio y valproato; si existe agitación psicomotora severa, coadyuvar con benzodiacepinas (clonazepam/lorazepam) hasta el efecto antimaníaco pleno.

**4. Alertas de Seguridad y Trazabilidad Comercial:**
*   **ALERTA DE CASCADA DE PRESCRIPCIÓN:** El uso de antidepresivos en monoterapia en pacientes con trastorno bipolar es contraindicado por riesgo de viraje maníaco, ciclación rápida y agravamiento del riesgo suicida. La desescalada inmediata al estabilizador es obligatoria.
*   **Trazabilidad Comercial:** Los fármacos mencionados (Litio, Valproato, Quetiapina, Aripiprazol, Risperidona, Asenapina) son marcas comerciales de laboratorios como **Lemontec**, **Sandoz**, **Bayer** y **Pfizer**.
*   **Consideración Metabólica:** Se advierte sobre el riesgo de interacciones enzimáticas masivas (CYP3A4) con antipsicóticos, especialmente si se coadministra con otros fármacos.


[CASO 3] Verificación:
• Tasa de recuperación de esperados: 3/3 (100.0%) -> ['GUIA-CANMAT-MANIA-001', 'GUIA-CANMAT-DEP-002', 'ALERTA-ISRS-VIRAJE-004']
• Citas válidas formalmente indexadas: 0/0
✓ Todas las citas válidas corresponden a documentos presentes en el contexto.


100.0

### Caso 4: Detección de Cascada de Prescripción (Haloperidol y Biperideno) [CASC-HAL-BIP-001]
Evaluación del juicio clínico crítico para identificar cascadas iatrogénicas (tratamiento crónico de extrapiramidalismo con anticolinérgicos) y el algoritmo de desprescripción gradual.


In [12]:
pregunta_4 = "Un paciente en tratamiento con Haloperidol comienza con rigidez y temblor y se le prescribe biperideno de forma continuada. ¿Por qué constituye una cascada de prescripción iatrogénica y qué pauta de desprescripción gradual se aconseja?"
resp_4, docs_4 = generar_respuesta_rag(pregunta_4, vector_store)

from IPython.display import display, Markdown
print("=" * 70)
print("🤖 RESPUESTA EN VIVO DEL MOTOR DE IA SOFIA - CASO 4")
print("=" * 70)
display(Markdown(resp_4))
print("\n" + "=" * 60)
verificar_caso("CASO 4", pregunta_4, ["CASC-HAL-BIP-001", "MONO-HALOPERIDOL-008"], resp_4, docs_4, vector_store)


🤖 RESPUESTA EN VIVO DEL MOTOR DE IA SOFIA - CASO 4


**1. Juicio Clínico Inmediato:**
La combinación de Haloperidol con Biperideno en un paciente con rigidez y temblor constituye una **cascada de prescripción iatrogénica**, ya que el antipsicótico típico induce efectos anticolinérgicos que exacerban el parkinsonismo secundario, sumando riesgo cognitivo y discinesia tardía. La pauta recomendada es reducir la dosis de Haloperidol o rotar a un antipsicótico atípico con menor perfil extrapiramidal (ej. quetiapina u olanzapina).

**2. Mecanismo Farmacológico y Cinética:**
El **Haloperidol** actúa como antagonista potente de receptores dopaminérgicos D2 mesolímbicos y nigroestriatales, lo que explica el parkinsonismo secundario. La administración crónica de Biperideno (antagonista anticolinérgico terciario) induce una **carga anticolinérgica** significativa. Esto genera un efecto sinérgico iatrogénico: la reducción de dopamina en el sistema nigroestriatal se agrava por la inhibición colinérgica, exacerbando rigidez y temblor. Además, existe riesgo de **discinesia tardía irreversible** y prolongación del intervalo QTc debido a la combinación de ambos fármacos.

**3. Pauta Operativa y Recomendaciones:**
*   **Desprescripción Gradual:** No se recomienda rotar inmediatamente al Biperideno. Se debe reducir la dosis de Haloperidol en pasos pequeños (ej. 25-50% semanalmente) mientras se monitorea el parkinsonismo.
*   **Monitorización Obligatoria:** Vigilar signos de agranulocitosis, litemia y síndrome neuroléptico maligno.
*   **Alternativa:** Considerar la rotación a un antipsicótico atípico (ej. Quetiapina o Olanzapina) para evitar la cascada de prescripción con anticolinérgicos.

**4. Alertas de Seguridad y Trazabilidad Comercial:**
*   **ALERTA DE CASCADA DE PRESCRIPCIÓN:** La combinación Haloperidol + Biperideno es iatrogénica por efecto anticolinérgico sinérgico, exacerbando rigidez y temblor.
*   **Código de Citación:** [MONO-HALOPERIDOL-008], [MARCA-BIPERIDENO-001].
*   **Advertencia:** Rotar a antipsicóticos atípicos con menor perfil extrapiramidal (ej. Quetiapina) para evitar la cascada de prescripción iatrogénica.


[CASO 4] Verificación:
• Tasa de recuperación de esperados: 2/2 (100.0%) -> ['CASC-HAL-BIP-001', 'MONO-HALOPERIDOL-008']
• Citas válidas formalmente indexadas: 2/2
✓ Todas las citas válidas corresponden a documentos presentes en el contexto.


100.0

## 7. Evaluación Formal del Motor RAG: Benchmark Clínico, Recall@k y MRR

Para garantizar la solidez y reproducibilidad médica del sistema, evaluamos el desempeño del recuperador clínico sobre un **conjunto de prueba de 15 consultas médicas representativas**, midiendo métricas formales de Information Retrieval (IR):
- **Recall@k:** Proporción de documentos clínicos esperados recuperados en los primeros $k$ resultados.
- **MRR (Mean Reciprocal Rank):** Posición recíproca del primer documento relevante en la lista recuperada.


In [13]:
# Benchmark clínico de evaluación cuantitativa del RAG
BENCHMARK_CLINICO_SOFIA = [
    {'id': 'T01', 'pregunta': '¿Qué laboratorios comercializan la Sertralina y cuáles son sus marcas comerciales?', 'docs_esperados': ['MARCA-SERTRALINA-001']},
    {'id': 'T02', 'pregunta': 'Interacciones del citocromo CYP2D6 con Paroxetina y Bupropión', 'docs_esperados': ['INTER-CYP2D6-001', 'MONO-PAROXETINA-003', 'MONO-BUPROPION-004']},
    {'id': 'T03', 'pregunta': 'Tratamiento de primera línea en manía aguda según guías CANMAT', 'docs_esperados': ['GUIA-CANMAT-MANIA-001']},
    {'id': 'T04', 'pregunta': 'Cascada de prescripción de haloperidol con biperideno parkinsonismo secundario', 'docs_esperados': ['CASC-HAL-BIP-001', 'MONO-HALOPERIDOL-008']},
    {'id': 'T05', 'pregunta': 'Alerta de viraje maníaco con antidepresivos en depresión bipolar CANMAT', 'docs_esperados': ['ALERTA-ISRS-VIRAJE-004', 'GUIA-CANMAT-DEP-002']},
    {'id': 'T06', 'pregunta': 'Interacciones renales del Carbonato de Litio con AINEs y diuréticos', 'docs_esperados': ['INTER-LIT-RENAL-004', 'MONO-LITIO-005']},
    {'id': 'T07', 'pregunta': 'Riesgo de agranulocitosis por clozapina y controles hematológicos obligatorios', 'docs_esperados': ['ALERTA-CLOZ-AGRAN-001', 'MONO-CLOZAPINA-006']},
    {'id': 'T08', 'pregunta': 'Interacción farmacocinética entre lamotrigina y ácido valproico riesgo de Stevens-Johnson', 'docs_esperados': ['INTER-LAM-VAL-003', 'ALERTA-LAM-SJS-002']},
    {'id': 'T09', 'pregunta': 'Cascada metabólica por antipsicóticos atípicos olanzapina quetiapina ganancia de peso', 'docs_esperados': ['CASC-METAB-002']},
    {'id': 'T10', 'pregunta': 'Marcas comerciales y laboratorios de Carbonato de Litio en España', 'docs_esperados': ['MARCA-CARBONATO-DE-LITIO-001', 'MONO-LITIO-005']},
    {'id': 'T11', 'pregunta': 'Efecto del tabaco e hidrocarburos sobre el citocromo CYP1A2 y niveles de Clozapina', 'docs_esperados': ['INTER-CYP1A2-002', 'MONO-CLOZAPINA-006']},
    {'id': 'T12', 'pregunta': 'Posología, perfil clínico e indicaciones de Escitalopram', 'docs_esperados': ['JUFE-MONO-ESCITALOPRAM', 'STAHL-MONO-ESCITALOPRAM']},
    {'id': 'T13', 'pregunta': 'Recomendaciones de Quetiapina en monoterapia o combinada en manía bipolar CANMAT', 'docs_esperados': ['GUIA-CANMAT-REC-002', 'GUIA-CANMAT-MANIA-001']},
    {'id': 'T14', 'pregunta': 'Monografía de Venlafaxina mecanismo dual e interacciones', 'docs_esperados': ['JUFE-MONO-VENLAFAXINA', 'STAHL-MONO-VENLAFAXINA']},
    {'id': 'T15', 'pregunta': 'Marcas comerciales, laboratorios y presentaciones de Clonazepam', 'docs_esperados': ['MARCA-CLONAZEPAM-001']},
    # Consultas de control fuera de dominio (Out-of-Domain) para evaluar abstención
    {'id': 'OOD01', 'pregunta': '¿Cómo cocinar una tortilla de patatas con huevos y sal?', 'docs_esperados': [], 'debe_abstenerse': True},
    {'id': 'OOD02', 'pregunta': '¿Cómo regar los geranios y las orquídeas en primavera?', 'docs_esperados': [], 'debe_abstenerse': True},
    {'id': 'OOD03', 'pregunta': '¿Quién pintó la Mona Lisa en el Museo del Louvre de París?', 'docs_esperados': [], 'debe_abstenerse': True}
]

# Validación rigurosa de identificadores esperados contra la base cargada
ids_validos = {d['id'] for d in documentos_rag}
for caso in BENCHMARK_CLINICO_SOFIA:
    if not caso.get('debe_abstenerse', False):
        faltantes = set(caso['docs_esperados']) - ids_validos
        assert not faltantes, f"Error en {caso['id']}: IDs esperados inexistentes en el corpus: {faltantes}"

def evaluar_recuperacion_rag(vs, benchmark, k=5):
    recalls = []
    rr_list = []
    filas_reporte = []
    abstenciones_correctas = 0
    total_ood = 0

    print(f"Modo de índice de recuperación activo: '{vs.modo_indice}' (Embedding model: {vs.embedding_model if vs.use_ollama_embeddings else 'TF-IDF Léxico'})")

    for caso in benchmark:
        q = caso['pregunta']
        es_ood = caso.get('debe_abstenerse', False)
        esperados = sorted(set(caso['docs_esperados']))

        # Recuperar más fragmentos y deduplicar por ID de documento padre único
        recuperados_raw = vs.buscar(q, top_k=k * 3)
        ids_rec = list(dict.fromkeys(d['id'] for d in recuperados_raw))[:k]
        max_score = max([d['score_similitud'] for d in recuperados_raw]) if recuperados_raw else 0.0

        if es_ood:
            total_ood += 1
            # Comprobar si se abstiene correctamente según el umbral por modo
            resp, _ = generar_respuesta_rag(q, vs)
            abstuvo = "No consta" in resp
            if abstuvo:
                abstenciones_correctas += 1
            filas_reporte.append({
                'ID': caso['id'],
                'Consulta': (q[:40] + '...') if len(q) > 40 else q,
                'Esperados': '[Out-of-Domain]',
                f'Aciertos@{k}': 'N/A (Control)',
                f'Recall@{k}': 1.0 if abstuvo else 0.0,
                'MRR': 'Abstención OK' if abstuvo else 'Falso Positivo'
            })
            continue

        # Recall@k sobre documentos únicos
        hits = len(set(esperados).intersection(set(ids_rec)))
        recall = hits / len(esperados) if esperados else 0.0
        recalls.append(recall)

        # Reciprocal Rank (MRR)
        rr = 0.0
        for rank, d_id in enumerate(ids_rec, 1):
            if d_id in esperados:
                rr = 1.0 / rank
                break
        rr_list.append(rr)

        filas_reporte.append({
            'ID': caso['id'],
            'Consulta': (q[:40] + '...') if len(q) > 40 else q,
            'Esperados': ', '.join(esperados),
            f'Aciertos@{k}': f"{hits}/{len(esperados)}",
            f'Recall@{k}': round(recall, 2),
            'MRR': round(rr, 2)
        })

    df_resultado = pd.DataFrame(filas_reporte)
    media_recall = float(np.mean(recalls))
    media_mrr = float(np.mean(rr_list))

    print(f"\n=== EVALUACIÓN FORMAL DEL RECUPERADOR RAG (Benchmark: {len(benchmark)} consultas) ===")
    print(f"• Recall@{k} Global (Consultas Clínicas): {media_recall*100:.1f}%")
    print(f"• MRR (Mean Reciprocal Rank): {media_mrr:.3f}")
    print(f"• Tasa de Abstención en Out-of-Domain: {abstenciones_correctas}/{total_ood} ({(abstenciones_correctas/max(total_ood, 1))*100:.1f}%)")
    print("\nDetalle por Caso de Prueba:")
    print(df_resultado.to_string(index=False))

    return {'recall_medio': media_recall, 'mrr': media_mrr, 'tabla': df_resultado}

# Ejecución formal de la evaluación
res_eval = evaluar_recuperacion_rag(vector_store, BENCHMARK_CLINICO_SOFIA, k=5)


Modo de índice de recuperación activo: 'ollama' (Embedding model: embeddinggemma:latest)

=== EVALUACIÓN FORMAL DEL RECUPERADOR RAG (Benchmark: 18 consultas) ===
• Recall@5 Global (Consultas Clínicas): 97.8%
• MRR (Mean Reciprocal Rank): 0.922
• Tasa de Abstención en Out-of-Domain: 3/3 (100.0%)

Detalle por Caso de Prueba:
   ID                                    Consulta                                                 Esperados    Aciertos@5  Recall@5           MRR
  T01 ¿Qué laboratorios comercializan la Sertr...                                      MARCA-SERTRALINA-001           1/1      1.00           1.0
  T02 Interacciones del citocromo CYP2D6 con P... INTER-CYP2D6-001, MONO-BUPROPION-004, MONO-PAROXETINA-003           2/3      0.67          0.33
  T03 Tratamiento de primera línea en manía ag...                                     GUIA-CANMAT-MANIA-001           1/1      1.00           1.0
  T04 Cascada de prescripción de haloperidol c...                    CASC-HAL-BIP-001, MONO

## 8. Exportación de la Base de Conocimiento a JSON

Guardamos las fichas clínicas integradas en `data/processed/rag_knowledge_base.json` para que puedan ser cargadas directamente por la aplicación de **Streamlit**.


In [14]:
# Verificación y persistencia de la base de conocimiento consolidada
json_path = DATA_PROCESSED / 'rag_knowledge_base.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(documentos_rag, f, indent=2, ensure_ascii=False)

rel_json = json_path.relative_to(project_root)
assert len(documentos_rag) >= 200, f'Error: Base RAG incompleta ({len(documentos_rag)} docs)'
print(f'Base de conocimiento RAG validada y guardada con éxito ({len(documentos_rag)} documentos) en: {rel_json}')

Base de conocimiento RAG validada y guardada con éxito (244 documentos) en: data/processed/rag_knowledge_base.json
